In [1]:
!pip install confluent_kafka
!pip install pyspark==3.5.2
!pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 37.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 5.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.9 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812369 sha256=2fef9d0670ef36cc17d12266b2220a4cad1c32f3447b8a7b35504fcf0b139f00
  Stored in directory: /root/.cache/pip/wheels/cf/c0/b9/f147f4220fd1d9277d0981b88b35b26f03ad910fffd60013a6
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are in

In [32]:
# ============================================================
# TODO在这里: Sentiment Analysis 可以用多个模型来对比
# ============================================================

def analyze_sentiment(text):
    if not text:
        return "neutral"
    # TODO: Replace with actual sentiment analysis
    return "neutral"

sentiment_udf = udf(analyze_sentiment, StringType())


In [37]:
# # ============================================================
# # 清空 Checkpoint, 从这个目录是用来记录读到哪一条了
# # ============================================================
# import shutil

# # 删除整个 checkpoint 目录
# checkpoint_dir = "/kaggle/working/checkpoints"
# shutil.rmtree(checkpoint_dir, ignore_errors=True)

# # 也删掉旧的 checkpoint.txt（如果存在）
# import os
# if os.path.exists("/kaggle/working/checkpoint.txt"):
#     os.remove("/kaggle/working/checkpoint.txt")

In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from elasticsearch import Elasticsearch, helpers
import json
import os

# ============================================================
# Configuration
# ============================================================

config = {
    "kafka": {
        "bootstrap.servers": "pkc-ox31np.ap-southeast-7.aws.confluent.cloud:9092",
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms": "PLAIN",
        "sasl.username": "S4AU73FQKMOAMSPL",
        "sasl.password": "cfltEuKzwF9VDz4Y2XwilhMpyc+9rxUP4xYfpOFYbbC1VefzzNc8rOYHfIavyVLw",
    },
    "elasticsearch": {
        "host": "https://20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com:443",
        "api_key": "eEFjNGJaMEJfQXBJNldXZFc3YU06RWhYdmpGZ1lsNzZlYWJJa1QwMXdQdw==",
        "index": "enron_emails"
    }
}

# Checkpoint directory
checkpoint_dir = "/kaggle/working/checkpoints/enron_kafka_es"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# ============================================================
# Schema for Enron emails (matches your Kafka data)
# ============================================================

email_schema = StructType([
    StructField("file_path", StringType(), True),
    StructField("message_id", StringType(), True),
    StructField("sent_at", StringType(), True),
    StructField("sender", StringType(), True),
    StructField("to", StringType(), True),
    StructField("cc", StringType(), True),
    StructField("bcc", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("x_folder", StringType(), True),
    StructField("x_origin", StringType(), True),
    StructField("x_filename", StringType(), True),
    StructField("body", StringType(), True),
    StructField("body_length", IntegerType(), True)
])

In [40]:
# ============================================================
# 清空已有的 Elasticsearch 索引
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

if es_client.indices.exists(index=index_name):
    # 删除索引
    es_client.indices.delete(index=index_name)
    print(f"Deleted index: {index_name}")
else:
    print(f"Index {index_name} does not exist")


Deleted index: enron_emails


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [41]:

# ============================================================
# Elasticsearch client (for manual bulk writes)
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

# create index mapping
mapping = {
    "mappings": {
        "properties": {
            "bcc": {"type": "keyword"},
            "body": {"type": "text"},
            "body_length": {"type": "integer"},
            "cc": {"type": "keyword"},
            "file_path": {"type": "keyword"},
            "message_id": {"type": "keyword"},
            "sender": {"type": "keyword"},
            "sent_at": {
                "type": "date",
                "format": "yyyy-MM-dd HH:mm:ss||yyyy-MM-dd'T'HH:mm:ss.SSSZ||strict_date_optional_time||epoch_millis"
            },
            "sentiment": {"type": "keyword"},
            "subject": {
                "type": "text",
                "fields": {
                    "keyword": {
                        "type": "keyword",
                        "ignore_above": 256
                    }
                }
            },
            "to": {"type": "keyword"},
            "x_filename": {"type": "keyword"},
            "x_folder": {"type": "keyword"},
            "x_origin": {"type": "keyword"}
        }
    }
}

es_client.indices.create(index=index_name, body=mapping)

# # test
# test_doc = {
#     "sent_at": "2001-05-14 23:39:00",
#     "message_id": "test_123",
#     "sender": "test@enron.com",
#     "to": "test@enron.com",
#     "subject": "Test",
#     "body": "Test message",
#     "sentiment": "neutral"
# }

# result = es_client.index(index=index_name, document=test_doc, refresh=True)
# print(f"Test passed! Document ID: {result['_id']}")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'enron_emails'})

In [46]:
def write_to_es(df, epoch_id):
    if df.count() == 0:
        return
    
    # 转换 sent_at 日期格式
    if 'sent_at' in df.columns:
        from pyspark.sql.functions import regexp_replace, to_timestamp, date_format
        df = df.withColumn('sent_at', 
            regexp_replace(col('sent_at'), ' UTC$', ''))
        df = df.withColumn('sent_at', 
            to_timestamp(col('sent_at'), 'yyyy-MM-dd HH:mm:ss'))
        df = df.withColumn('sent_at',
            date_format(col('sent_at'), "yyyy-MM-dd'T'HH:mm:ss.SSSZ"))
    
    records = df.toJSON().collect()
    actions = []
    for record in records:
        actions.append({
            "_index": config['elasticsearch']['index'],
            "_source": json.loads(record)
        })
    
    if actions:
        # print(f"Sample record: {actions[0]['_source']}")
        
        success, failed = helpers.bulk(es_client, actions, stats_only=False, raise_on_error=False)
        print(f"Batch {epoch_id}: {success} succeeded, {len(failed)} failed")
        
        if failed:
            print(f"First failed record: {actions[0]['_source']}")
            print(f"Error details: {failed[0]}")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Sample record: {'file_path': 'bass-e/inbox/250.', 'message_id': '<9273402.1075859139444.JavaMail.evans@thyme>', 'sent_at': '2001-12-28T15:34:19.000+0000', 'sender': 'admin.enron@enron.com', 'to': 'eric.bass@enron.com', 'cc': '', 'bcc': '', 'subject': 'An Inbound Message For You Has Been Quarantined', 'x_folder': '\\Eric_Bass_Jan2002\\Bass, Eric\\Inbox', 'x_origin': 'Bass-E', 'x_filename': 'ebass (Non-Privileged).pst', 'body': "You have received this message because someone has attempted to send you an e-mail from outside of Enron with an attachment type that Enron does not allow into our messaging environment. Your e-mail has been quarantined and is being held at the MailSweeper server.\n\n\nSender:  daphneco64@alltel.net\nDate:  Fri, 28 Dec 2001 09:28:35 -0600\nSubject:  wedding expense who pays for what wedding hints . Country Weddings . wedding gu\nAttachment Type: Scenarios/Incoming/Inbound URL Catcher: A filename matching the file mask was detected: 'wedding expense who pays for w

In [43]:
# ============================================================
# Create Spark Session
# ============================================================

spark = SparkSession.builder \
    .appName("EnronKafkaToES") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2") \
    .getOrCreate()

# 设置日志级别为 ERROR 或 FATAL 来屏蔽 WARN
spark.sparkContext.setLogLevel("ERROR")

# ============================================================
# Read from Kafka and write to Elasticsearch
# ============================================================

def read_from_kafka_and_write_to_es(spark):
    topic = "raw_emails_topic"
    
    stream_df = (spark.readStream
                 .format("kafka")
                 .option("kafka.bootstrap.servers", config['kafka']['bootstrap.servers'])
                 .option("subscribe", topic)
                 .option("startingOffsets", "earliest")
                 .option("kafka.security.protocol", config['kafka']['security.protocol'])
                 .option("kafka.sasl.mechanism", config['kafka']['sasl.mechanisms'])
                 .option("kafka.sasl.jaas.config",
                        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{config["kafka"]["sasl.username"]}" '
                        f'password="{config["kafka"]["sasl.password"]}";')
                 .option("failOnDataLoss", "false")
                 .option("maxOffsetsPerTrigger", 100)
                 .load()
                )
    
    # Parse JSON
    parsed_df = stream_df.select(
        from_json(col('value').cast("string"), email_schema).alias("data")
    ).select("data.*")
    
    # Add sentiment (placeholder)
    enriched_df = parsed_df.withColumn("sentiment", sentiment_udf(col("body")))
    
    # Write to Elasticsearch via foreachBatch
    query = (enriched_df.writeStream
             .foreachBatch(write_to_es)
             .outputMode("append")
             .trigger(processingTime="10 seconds")
             .option("checkpointLocation", checkpoint_dir)
             .start()
             .awaitTermination()
            )

In [47]:
# 一直运行就会一直流处理    
if __name__ == "__main__":
    read_from_kafka_and_write_to_es(spark)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 163: 96 succeeded, 0 failed


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 164: 96 succeeded, 0 failed


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 165: 96 succeeded, 0 failed


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [48]:
# 查询最新的5条记录
response = es_client.search(
    index="enron_emails",
    body={
        "query": {"match_all": {}},
        "sort": [{"sent_at": {"order": "desc"}}],
        "size": 5
    }
)

# 打印结果
print(f"Total records: {response['hits']['total']['value']}")
print("\n=== Latest 5 Records ===\n")
for i, hit in enumerate(response['hits']['hits'], 1):
    source = hit['_source']
    print(f"{i}. ID: {hit['_id']}")
    print(f"   Sent at: {source.get('sent_at')}")
    print(f"   From: {source.get('sender')}")
    print(f"   To: {source.get('to')}")
    print(f"   Subject: {source.get('subject', '')[:50]}")
    print(f"   Sentiment: {source.get('sentiment')}")
    print("-" * 50)

Total records: 10000

=== Latest 5 Records ===

1. ID: IQeYbZ0B_ApI6WWdbfgc
   Sent at: 2004-02-04T01:45:38.000+0000
   From: 1800flowers.224433405@s2u2.com
   To: ebass@enron.com
   Subject: SAVE $20* while sending gifts to everyone on your 
   Sentiment: neutral
--------------------------------------------------
2. ID: JAeVbZ0B_ApI6WWdtvRl
   Sent at: 2004-02-04T01:45:35.000+0000
   From: 1800flowers.238953685@s2u2.com
   To: ebass@enron.com
   Subject: Treat yourself to savings in our Post-Holiday Sale
   Sentiment: neutral
--------------------------------------------------
3. ID: sDWPbZ0BpeIanCvL6D6p
   Sent at: 2002-03-21T18:48:12.000+0000
   From: sbailey@crusescott.com
   To: susan.bailey@enron.com
   Subject: FW: Strawnie
   Sentiment: neutral
--------------------------------------------------
4. ID: AjWPbZ0BpeIanCvLmj5Y
   Sent at: 2002-03-21T18:11:22.000+0000
   From: susan.bailey@enron.com
   To: louis.dicarlo@enron.com
   Subject: RE: EOTT Energy Partners LP
   Sentiment: n

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 166: 96 succeeded, 0 failed
